# 📘 코드 해설판 (Python 초보자용)

이 노트북은 원본 `1. Llama_index.ipynb` 에 **번호(`# 1`, `# 2` …)가 붙은 코드 셀마다 한국어 해설**을 덧붙인 파일입니다.
코드 셀 바로 **위**에 있는 `💡 [# N] 코드 설명` 칸이 해설입니다. 원본 코드와 원본 설명은 그대로 두었습니다.

**이 노트북에서 배우는 것 (큰 그림)**

| 용어 | 쉬운 뜻 |
|---|---|
| LLM | ChatGPT 같은 "글을 만들어 내는 AI 모델" |
| RAG | AI가 답하기 전에 **관련 문서를 먼저 찾아보고(Retrieval)** 그 내용을 참고해 답을 만드는(Generation) 방식 |
| LlamaIndex | RAG를 쉽게 만들도록 도와주는 Python 라이브러리 |
| Document | 불러온 원본 문서 한 개 |
| Node(Chunk) | 문서를 잘게 자른 조각 |
| Embedding | 글을 숫자 목록(벡터)으로 바꾼 것. 뜻이 비슷한 글은 숫자도 비슷해짐 |
| Index | 조각들과 그 임베딩을 모아 둔 "검색용 색인" |
| Retriever | 질문과 비슷한 조각을 찾아오는 "검색 담당" |
| Query Engine | 검색 + 답변 생성을 한 번에 해 주는 "질문 창구" |

흐름: **문서 불러오기 → 조각내기 → 임베딩 → 인덱스 만들기 → 질문하기 → 문서 추가/수정/삭제 → 나만의 Query Engine 만들기**

# 📓 Llama-Index Practice

In this section, we will create a simple Llama Index app and learn how to log and get feedback on an LLM response. Additionally, we will explore how the query engine can be modified based on its intended purpose.​

Through this process, we hope to gain a clear understanding of the role and functionalities of the Llama Index, as well as the specific inputs and outputs of each method.  

<img src="https://miro.medium.com/v2/resize:fit:1400/0*CrywD0tloiK9dgy_.png" width="600">

This section is divided into three main stages:

###I. Build a Query Engine  
###II. Use Database Management Method  
###III. Make a Custom Query Engine

Let's first look at what is needed to build a Query Engine.

## I. Buliding a Query Engine

The task of this section is to create a Query Engine that takes a user's query, retrieves for related content, and returns a final summary.  

Ignore the `Vector DB` in the picture. We don't use external vector database in the pracetice. Instead, we will use local memory as simple vector store.


<img src="https://i.imgur.com/bR4xaBd.png">

To accomplish this task, the following steps will be taken.
<br/>

#### 1. Install and Import Libraries
#### 2. DownLoad Data
#### 3. Make Query Engine from Index Directly
#### 4. Conduct Activity


### 1. Install and Import Libraries

### 💡 [# 1] 코드 설명 — 필요한 라이브러리 설치

**한 줄 요약:** 이 실습에 필요한 프로그램 꾸러미(라이브러리)를 내 컴퓨터에 설치합니다.

- `import os` : 운영체제(폴더, 환경변수 등)를 다루는 파이썬 기본 도구를 불러옵니다.
- `os.environ.pop("PYTHONPATH", None)` : `os.environ` 은 "환경변수 사전"입니다. 여기서 `PYTHONPATH` 라는 설정을 꺼내서 지웁니다. 다른 곳에 깔린 라이브러리와 섞여 충돌하는 것을 막기 위한 안전장치입니다. 뒤의 `None` 은 "그 항목이 없어도 오류 내지 말라"는 뜻입니다.
- `! pip install ...` : 줄 맨 앞의 `!` 는 파이썬 코드가 아니라 **터미널 명령어**를 실행하라는 주피터 전용 기호입니다. `pip install` 은 라이브러리 설치 명령입니다.
  - `llama_index==0.12.2` : `==` 뒤 숫자는 **버전 고정**입니다. 강의와 똑같은 버전을 써야 같은 결과가 나옵니다.
  - `openai` : OpenAI(ChatGPT) API를 쓰기 위한 라이브러리
  - `--quiet` : 설치 과정 메시지를 줄여서 조용히 설치합니다.

> 🔰 **초보자 팁:** 설치는 한 번만 하면 됩니다. 설치 후 오류가 나면 메뉴에서 **커널 재시작(Restart Kernel)** 후 다음 셀부터 실행해 보세요.
> 참고로 `packaging>=24.0` 의 `>` 기호는 터미널에서 "파일로 저장"이라는 뜻으로 해석될 수 있어, 폴더에 `=24.0` 이라는 빈 파일이 생길 수 있습니다. 해가 없으니 지워도 됩니다.

In [2]:
# 1
### YOUR CODE HERE ###
import os
os.environ.pop("PYTHONPATH", None)

! pip install llama_index==0.12.2 openai==1.55.3 packaging>=24.0 streamlit==1.35.0 --quiet

### 💡 [# 2] 코드 설명 — API 키 등록과 도구 불러오기

**한 줄 요약:** OpenAI를 쓸 수 있는 "비밀번호(API 키)"를 등록하고, 앞으로 쓸 도구들을 불러옵니다.

- `os.environ["OPENAI_API_KEY"] = "sk-..."` : 환경변수에 API 키를 저장합니다. OpenAI·LlamaIndex 라이브러리는 이 이름의 환경변수를 자동으로 찾아 씁니다.
- `from A import B` : "A라는 꾸러미에서 B만 꺼내 쓰겠다"는 문법입니다.
  - `VectorStoreIndex` : 문서를 임베딩해 저장하는 **검색용 색인**
  - `SimpleDirectoryReader` : 폴더 안의 파일을 읽어 문서로 만들어 주는 도구
  - `Document` : 문서 한 개를 담는 그릇
  - `PromptTemplate` : AI에게 보낼 지시문(프롬프트)의 **틀**
  - `CustomQueryEngine`, `BaseRetriever`, `BaseSynthesizer`, `get_response_synthesizer` : 뒤쪽(III장)에서 나만의 Query Engine을 만들 때 쓰는 부품들
- `import textwrap` : 긴 글을 보기 좋게 줄바꿈해 주는 기본 도구
- `import openai` : OpenAI API를 직접 호출할 때 사용

> 🔰 **초보자 팁:** API 키는 **신용카드 번호처럼 비밀**입니다. 이 노트북 파일을 남에게 공유하거나 GitHub에 올릴 때는 키를 반드시 지우세요.

In [3]:
# 2
### YOUR CODE HERE ###

import os
os.environ["OPENAI_API_KEY"] = "sk-..." #Insert your openai api key

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Document, PromptTemplate
from openai import OpenAI

from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.retrievers import BaseRetriever
from llama_index.core import get_response_synthesizer
from llama_index.core.response_synthesizers import BaseSynthesizer

import textwrap
import openai

C:\Users\swsuser-j29\.conda\envs\rag\lib\site-packages\pydantic\_internal\_generate_schema.py:2274: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


### 2. DownLoad Data

Previously, we learned that the Query Engine receives the following elements as input and output.
<br/>
#### Input: **User Query**, **Vector Database**
#### Output: **Summary of Retrieved Passages**
<br/>

Here, the User Query is an element that we can decide. However, since the Vector Database must contain information related to the Query, we need to consider how to configure the Vector Database before proceeding with the practice.

The Vector Database can be divided into two categories: a storage that allows the query engine to retrieve related passages, and the data containing the related passages. In this practice, we will use local memory as a data storage instead of an external Vector Database, so the Vector Database will be referred to as a Vector Store from now on. For the data, we will use a simple txt file provided by Llama Index.

Before we begin the practical exercises, let's first take a look at the txt file we used. The code below can be used to download the txt file we used with the `wget` command.


### 💡 [# 3] 코드 설명 — 실습용 글(데이터) 내려받기

**한 줄 요약:** 인터넷에서 Paul Graham의 에세이 텍스트 파일을 받아 `data` 폴더에 저장합니다.

- `import os, urllib.request` : 쉼표로 여러 도구를 한 번에 불러옵니다. `urllib.request` 는 인터넷에서 파일을 받는 기본 도구입니다.
- `os.makedirs("data", exist_ok=True)` : `data` 라는 폴더를 만듭니다. `exist_ok=True` 는 "이미 있어도 오류 내지 말라"는 뜻입니다.
- `url = "https://..."` : 받을 파일의 인터넷 주소를 `url` 이라는 변수(이름표)에 저장합니다.
- `urllib.request.urlretrieve(url, "data/paul_graham_essay.txt")` : 그 주소의 파일을 내려받아 `data/paul_graham_essay.txt` 라는 이름으로 저장합니다.

> 🔰 실행 결과로 `('data/paul_graham_essay.txt', ...)` 같은 것이 보이면 성공입니다.

In [4]:
# 3
### YOUR CODE HERE ###

import os, urllib.request

os.makedirs("data", exist_ok=True)
url = "https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/paul_graham/paul_graham_essay.txt"
urllib.request.urlretrieve(url, "data/paul_graham_essay.txt")

('data/paul_graham_essay.txt', <http.client.HTTPMessage at 0x183f92f01c0>)

If there are no issues with the download, you should be able to see that the `paul_graham_essay.txt` file has been downloaded.

This example uses the text of Paul Graham’s essay, [“What I Worked On”](https://paulgraham.com/worked.html), and is the canonical llama-index example.


Let’s print the first fifteen lines of the txt file. This will allow us to see that the text file deals with the experiences and reflections of one person's life.


### 💡 [# 4] 코드 설명 — 지금 내가 있는 폴더 확인

**한 줄 요약:** 노트북이 현재 "어느 폴더에서" 실행되고 있는지 출력합니다.

- `os.getcwd()` : **c**urrent **w**orking **d**irectory(현재 작업 폴더)의 경로를 알려 줍니다.
- `print(...)` : 괄호 안의 값을 화면에 출력합니다.

> 🔰 다음 셀에서 파일 경로를 직접 적어야 하는데, 여기서 나온 경로를 보고 맞춰 적으면 됩니다.

In [5]:
# 4
import os
print(os.getcwd())

C:\Users\RAG\1일차\실습 자료\Code


### 💡 [# 5] 코드 설명 — 받은 파일의 앞부분 15줄 읽어 보기

**한 줄 요약:** 텍스트 파일을 열어 처음 15줄만 화면에 보여 줍니다.

- `path_to_txt = r'C:\...'` : 파일 위치를 변수에 저장합니다. 따옴표 앞의 `r` 은 "raw 문자열"로, 윈도우 경로의 `\` 를 특수문자로 오해하지 않게 해 줍니다. **본인 컴퓨터 경로로 바꿔야 합니다.**
- `with open(path_to_txt, 'r', encoding='utf-8') as file:` : 파일을 읽기 모드(`'r'`)로 엽니다. `with` 를 쓰면 작업이 끝날 때 파일을 **자동으로 닫아** 줍니다. `as file` 은 열린 파일을 `file` 이라고 부르겠다는 뜻입니다.
- `for i in range(15):` : 아래 들여쓴 코드를 15번 반복합니다.
- `line = file.readline()` : 파일에서 **한 줄**을 읽습니다.
- `if not line: break` : 더 읽을 줄이 없으면(빈 값이면) 반복을 멈춥니다.
- `print(line.strip())` : `.strip()` 으로 양끝의 공백·줄바꿈을 떼고 출력합니다.
- `print("\n...")` : `\n` 은 줄바꿈입니다. "뒤에 더 있다"는 표시로 `...` 을 찍습니다.

In [7]:
# 5
### YOUR CODE HERE ###

path_to_txt = r'C:\Users\RAG\1일차\실습 자료\Code\data\paul_graham_essay.txt' #change this path

with open(path_to_txt, 'r', encoding='utf-8') as file:
    for i in range(15):
        line = file.readline()
        if not line:
            break
        print(line.strip())
print("\n...")



What I Worked On

February 2021

Before college the two main things I worked on, outside of school, were writing and programming. I didn't write essays. I wrote what beginning writers were supposed to write then, and probably still are: short stories. My stories were awful. They had hardly any plot, just characters with strong feelings, which I imagined made them deep.

The first programs I tried writing were on the IBM 1401 that our school district used for what was then called "data processing." This was in 9th grade, so I was 13 or 14. The school district's 1401 happened to be in the basement of our junior high school, and my friend Rich Draves and I got permission to use it. It was like a mini Bond villain's lair down there, with all these alien-looking machines — CPU, disk drives, printer, card reader — sitting up on a raised floor under bright fluorescent lights.

The language we used was an early version of Fortran. You had to type programs on punch cards, then stack them in t

### 3. Make Query Engine from Index Directly

Now let's create a query engine unsing the above information.

First, since Llama Index cannot directly handle text files in their raw form, we must convert them into Documents and Nodes. This can be accomplished with the following code.

```Python
documents = SimpleDirectoryReader("data").load_data()
```

SimpleDirectoryReader takes the directory path containing data that will go into the Vector Store as input and converts all the files within that directory into Document objects. The data types that can be converted into Documents are as follows.


*   .txt - text file  
*   .csv - comma-separated values  
*   .docx - Microsoft Word  
*   .epub - EPUB ebook format  
*   .hwp - Hangul Word Processor  
*   .ipynb - Jupyter Notebook  
*   .jpeg, .jpg - JPEG image  
*   .mbox - MBOX email archive  
*   .md - Markdown  
*   .mp3, .mp4 - audio and video  
*   .pdf - Portable Document Format  
*   .png - Portable Network Graphics  
*   .ppt, .pptm, .pptx - Microsoft PowerPoint  

You can view the text contained in the `documents` using the following command:

```Python
for i in range(len(documents)):
  print(documents[i].text)
```

### 💡 [# 6] 코드 설명 — 폴더의 파일을 Document로 불러오기

**한 줄 요약:** `data` 폴더 안의 모든 파일을 읽어 LlamaIndex가 이해하는 `Document` 목록으로 만듭니다.

- `SimpleDirectoryReader("data")` : `data` 폴더를 읽을 준비를 합니다.
- `.load_data()` : 실제로 파일을 읽어 `Document` 들의 **리스트(목록)** 를 돌려줍니다.
- `documents = ...` : 그 결과를 `documents` 라는 변수에 담습니다.

> 🔰 지금은 파일이 1개라서 `documents` 안에도 문서가 1개 들어 있습니다.

In [8]:
# 6
### YOUR CODE HERE ###

documents = SimpleDirectoryReader("data").load_data()

### 💡 [# 7] 코드 설명 — 불러온 문서 내용 출력

**한 줄 요약:** `documents` 안의 문서를 하나씩 꺼내 글 내용을 출력합니다.

- `len(documents)` : 목록 안에 몇 개가 들어 있는지 셉니다.
- `range(len(documents))` : 0부터 (개수-1)까지의 숫자를 차례로 만들어 줍니다.
- `documents[i]` : 목록에서 i번째 문서를 꺼냅니다. (파이썬은 **0번부터** 셉니다.)
- `.text` : 그 문서의 글 내용입니다.

> 🔰 에세이 전체가 출력되므로 결과가 매우 깁니다.

In [9]:
# 7
### YOUR CODE HERE ###

for i in range(len(documents)):
  print(documents[i].text)



What I Worked On

February 2021

Before college the two main things I worked on, outside of school, were writing and programming. I didn't write essays. I wrote what beginning writers were supposed to write then, and probably still are: short stories. My stories were awful. They had hardly any plot, just characters with strong feelings, which I imagined made them deep.

The first programs I tried writing were on the IBM 1401 that our school district used for what was then called "data processing." This was in 9th grade, so I was 13 or 14. The school district's 1401 happened to be in the basement of our junior high school, and my friend Rich Draves and I got permission to use it. It was like a mini Bond villain's lair down there, with all these alien-looking machines — CPU, disk drives, printer, card reader — sitting up on a raised floor under bright fluorescent lights.

The language we used was an early version of Fortran. You had to type programs on punch cards, then stack them in t

Once the Document is complete, we need to use this data to create an index.

The index is created using the VectorStoreIndex.from_documents() method. This method takes a list of Node objects or a Document as input.


```Python
index = VectorStoreIndex.from_documents(documents)
```

If connected to an external vector database, Llama Index follows the index algorithm of that vector database. If there is no connection to an external vector database, it stores the data in the local memory in the form of a dictionary.

### 💡 [# 8] 코드 설명 — 인덱스(검색용 색인) 만들기

**한 줄 요약:** 문서를 조각내고, 각 조각을 임베딩(숫자)으로 바꿔 검색할 수 있는 인덱스를 만듭니다.

- `VectorStoreIndex.from_documents(documents)` : 이 한 줄 안에서 아래 일이 자동으로 일어납니다.
  1. 문서를 적당한 크기의 조각(Node)으로 자름
  2. 각 조각을 OpenAI 임베딩 모델에 보내 숫자 벡터로 변환 (**이때 API 비용이 조금 발생**)
  3. 조각과 벡터를 메모리에 저장

> 🔰 비유: 책(문서)을 문단 카드(조각)로 나눈 뒤, 각 카드에 "뜻을 나타내는 좌표"를 붙여 서랍(인덱스)에 정리하는 과정입니다.

In [10]:
# 8
### YOUR CODE HERE ###

index = VectorStoreIndex.from_documents(documents)

Now, let’s look at how the index is structured.  

Our index is built based on documents. However, as mentioned in the previous presentation, when Llamaindex stores data in the database, it breaks down the document into nodes. Therefore, let’s first check if the document has been divided into nodes.

### 💡 [# 9] 코드 설명 — 인덱스 안의 조각(Node) 살펴보기

**한 줄 요약:** 인덱스에 조각이 몇 개 들어 있는지, 각 조각의 ID가 무엇인지 확인합니다.

- `index.index_struct.nodes_dict` : 인덱스가 가진 "조각 ID 사전(dict)"입니다. 사전은 `{키: 값}` 쌍의 모음입니다.
- `for key, value in node_id.items():` : 사전에서 (키, 값) 쌍을 하나씩 꺼냅니다.
- `node_example_id = value` 후 `break` : **첫 번째 조각의 ID만** 저장해 두고 반복을 바로 끝냅니다. (다음 셀에서 사용)
- `len(node_id.values())` : 조각의 총 개수
- `print(node_id.values())` : 모든 조각 ID를 출력

In [11]:
# 9
### YOUR CODE HERE ###

node_id = index.index_struct.nodes_dict

for key, value in node_id.items():
    print(key, value)
    node_example_id = value
    break

print("The number of nodes: ", len(node_id.values()))

print(node_id.values())

5b376492-ec69-49ec-b208-59d9cbf34567 5b376492-ec69-49ec-b208-59d9cbf34567
The number of nodes:  22
dict_values(['5b376492-ec69-49ec-b208-59d9cbf34567', '7ebcb294-7783-460e-a64a-b98f96e1b2a4', '55e33c0e-a7b9-4c77-9e28-b6a376f836af', '2ac228fb-7b52-4bc9-9c13-2d3ccb30d666', 'be0689b8-2f13-4568-b111-a3782b2f337e', 'aa257eb1-ae21-4dad-b36d-c561759434f0', '24c70e8a-b3b5-448f-9fd0-0636921abbba', 'e7e90435-6960-4877-a318-2a978d2abce3', 'db43f002-4756-4e75-ac5c-ff7db4e259cc', '7f11849f-9657-4e89-8e94-28f75b18ca00', '70d08ac4-0c7a-4773-96ac-119481cacda9', '527e7aef-fef1-49eb-b5da-e1dff342aaf4', '64e495b4-e92c-4f6f-a9fd-178081640495', '2c008483-945b-4a9e-9f6e-8e8fd2134236', '55abac53-0fee-4f30-b1a6-a1a15601770d', '86133739-69b7-4dc8-9fe1-c0e5aa8db8c5', 'c930bedd-a5cc-4bd4-89d7-af8e1f1d0db8', '8024954a-7a8d-4fc3-99a6-7da745b83b67', '04ca27e9-cc3b-4d26-840a-fbfc21a4acf4', '96be7e01-e53d-4c34-babe-757c8f8bad10', '46291b5b-f79b-4612-9b59-6b749613e58e', '8157463b-20ca-4d70-9571-42f1506bb5d2'])


By looking at this, we can check the id of the first node and the number of nodes the document has been divided into.

Next, let’s see how each node has been transformed into embeddings.


### 💡 [# 10] 코드 설명 — 조각 하나의 임베딩 값 보기

**한 줄 요약:** 앞에서 저장한 첫 번째 조각이 어떤 숫자 벡터로 바뀌었는지 확인합니다.

- `index.vector_store.data.embedding_dict` : "조각 ID → 임베딩 벡터" 사전입니다.
- `[node_example_id]` : 그 사전에서 해당 ID의 값을 꺼냅니다.

> 🔰 결과로 소수 1,536개가 쭉 나옵니다. 사람은 읽을 수 없지만, 컴퓨터는 이 숫자들이 **가까운지 먼지**를 계산해 "뜻이 비슷한 글"을 찾습니다. 셀의 마지막 줄에 값만 적으면 `print` 없이도 결과가 표시됩니다(주피터 기능).

In [12]:
# 10
### YOUR CODE HERE ###

index.vector_store.data.embedding_dict[node_example_id]

[0.0013735159300267696,
 -0.016709106042981148,
 -0.0041600861586630344,
 -0.033830780535936356,
 0.0041944668628275394,
 0.02835734561085701,
 -0.02683083526790142,
 -0.012535267509520054,
 -0.020518505945801735,
 -0.026500778272747993,
 0.016984151676297188,
 0.031465377658605576,
 -0.0073368786834180355,
 -0.009949825704097748,
 0.000671716348733753,
 0.01978963240981102,
 0.02263636887073517,
 0.014082406647503376,
 -0.001351168379187584,
 -0.017231695353984833,
 -0.01428869180381298,
 -0.0075844209641218185,
 0.010424281470477581,
 -0.002710931934416294,
 0.004053505137562752,
 0.0008500671247020364,
 0.019459575414657593,
 -0.037103839218616486,
 0.002530432539060712,
 -0.02014719322323799,
 0.018689444288611412,
 -0.01805683597922325,
 -0.0009085146593861282,
 -0.005810367874801159,
 -0.039056673645973206,
 -0.005370292812585831,
 -0.006642385385930538,
 -0.0012763900449499488,
 0.008093258365988731,
 -0.01399989239871502,
 0.03627869859337807,
 0.01888197660446167,
 0.004624227

As you can see here, each node is automatically converted into embeddings, becoming vectors when the index is created.

Next, let’s verify why the total number of nodes is as shown. When splitting a document into nodes, Llamaindex uses a class called SentenceSplitter. This class allows us to divide the text data into predefined lengths.

### 💡 [# 11] 코드 설명 — 문서를 직접 조각내 보기

**한 줄 요약:** `SentenceSplitter` 로 문서를 직접 잘라 보고, 조각이 몇 개 나오는지 셉니다.

- `SentenceSplitter(chunk_size=1024, chunk_overlap=200)` : 문장 경계를 살려서 자르는 도구입니다.
  - `chunk_size=1024` : 한 조각의 최대 크기(**토큰** 1024개). 토큰은 단어보다 조금 작은 글 단위입니다.
  - `chunk_overlap=200` : 이웃한 조각끼리 200토큰씩 **겹치게** 자릅니다. 문맥이 조각 경계에서 끊기는 것을 줄여 줍니다.
- `parser.get_nodes_from_documents(documents)` : 실제로 잘라서 조각 목록(`nodes`)을 만듭니다.
- `print(len(nodes))` : 조각 개수 출력. 이 값은 LlamaIndex 기본 설정이라 # 9에서 본 개수와 같습니다.

In [13]:
# 11
### YOUR CODE HERE ###

from llama_index.core.node_parser import SentenceSplitter

parser = SentenceSplitter(chunk_size=1024, chunk_overlap=200)
# you can change chunk_size, chunk_overlap

nodes = parser.get_nodes_from_documents(documents)

print(len(nodes))

22


Each node has an attribute called `text`, allowing us to see what text data each node contains. Additionally, you can see that the texts overlap by the value of the `chunk_overlap`.

When checking the length of each text data, we can see that they are composed of different numbers of characters. This is because the `SentenceSplitter` class divides `documents` not by the number of characters but by tokens, which are a type of sub-word.

### 💡 [# 12] 코드 설명 — 첫 두 조각 비교하기

**한 줄 요약:** 0번, 1번 조각의 글과 글자 수를 출력해 겹치는 부분(overlap)을 눈으로 확인합니다.

- `nodes[0].text`, `nodes[1].text` : 첫 번째와 두 번째 조각의 글
- `print("\n----------------------\n")` : 구분선 출력
- `len(nodes[0].text)` : 글자 수. `print` 에 쉼표로 여러 값을 주면 나란히 출력됩니다.

> 🔰 0번 조각의 **끝부분**과 1번 조각의 **앞부분**이 똑같은지 찾아보세요. 그것이 `chunk_overlap` 입니다.

In [14]:
# 12
### YOUR CODE HERE ###

print(nodes[0].text)
print("\n----------------------\n")
print(nodes[1].text)
print("\n----------------------\n")
print(len(nodes[0].text), len(nodes[1].text))

What I Worked On

February 2021

Before college the two main things I worked on, outside of school, were writing and programming. I didn't write essays. I wrote what beginning writers were supposed to write then, and probably still are: short stories. My stories were awful. They had hardly any plot, just characters with strong feelings, which I imagined made them deep.

The first programs I tried writing were on the IBM 1401 that our school district used for what was then called "data processing." This was in 9th grade, so I was 13 or 14. The school district's 1401 happened to be in the basement of our junior high school, and my friend Rich Draves and I got permission to use it. It was like a mini Bond villain's lair down there, with all these alien-looking machines — CPU, disk drives, printer, card reader — sitting up on a raised floor under bright fluorescent lights.

The language we used was an early version of Fortran. You had to type programs on punch cards, then stack them in the

We can use the default tokenizer used by Llamaindex’s `SentenceSplitter` to directly check how many tokens each text consists of.

By running the code below, we can bring in the tokenizer of `gpt-3.5-turbo`, which is used by default in `SentenceSplitter`, and perform tokenization to see the results and check how many tokens are generated.

When we look at the tokenization results, we will see that they differ from the `chunk_size` we set. This is because, when creating nodes, the amount of text data to be tokenized is determined by considering the metadata as well, in order to compose nodes of consistent size including the amount of metadata.

### 💡 [# 13] 코드 설명 — 토큰(token) 개수 세어 보기

**한 줄 요약:** 조각의 글을 토큰 번호로 바꿨다가 다시 글로 되돌려 보면서 "토큰"이 무엇인지 확인합니다.

- `import tiktoken` : OpenAI 모델이 쓰는 토큰 분리기 라이브러리
- `tiktoken.encoding_for_model("gpt-3.5-turbo")` : 해당 모델용 분리기를 가져옵니다.
- `tokenizer.encode(글)` : 글 → **숫자(토큰 번호) 목록**
- `len(encoded_text)` : 토큰 개수. 조각 크기 1024는 글자 수가 아니라 이 **토큰 수** 기준입니다.
- `tokenizer.decode(숫자목록)` : 숫자 목록 → 다시 글
- `print(f"...{변수}...")` : 문자열 앞에 `f` 를 붙이면 `{ }` 안의 변수값이 글 속에 끼워 넣어집니다(**f-string**).

In [15]:
# 13
### YOUR CODE HERE ###

import tiktoken
tokenizer = tiktoken.encoding_for_model("gpt-3.5-turbo")

encoded_text = tokenizer.encode(nodes[0].text)
print(f"Encoded text: {encoded_text}")
print(f"Encoded text length: {len(encoded_text)}")

decoded_text = tokenizer.decode(encoded_text)
print(f"Decoded text: \n\n{decoded_text}")

Encoded text: [3923, 358, 5664, 291, 1952, 271, 33877, 220, 2366, 16, 271, 10438, 7926, 279, 1403, 1925, 2574, 358, 6575, 389, 11, 4994, 315, 2978, 11, 1051, 4477, 323, 15840, 13, 358, 3287, 956, 3350, 23691, 13, 358, 6267, 1148, 7314, 16483, 1051, 10171, 311, 3350, 1243, 11, 323, 4762, 2103, 527, 25, 2875, 7493, 13, 3092, 7493, 1051, 25629, 13, 2435, 1047, 20781, 904, 7234, 11, 1120, 5885, 449, 3831, 16024, 11, 902, 358, 35706, 1903, 1124, 5655, 382, 791, 1176, 7620, 358, 6818, 4477, 1051, 389, 279, 29022, 220, 6860, 16, 430, 1057, 2978, 9474, 1511, 369, 1148, 574, 1243, 2663, 330, 695, 8863, 1210, 1115, 574, 304, 220, 24, 339, 12239, 11, 779, 358, 574, 220, 1032, 477, 220, 975, 13, 578, 2978, 9474, 596, 220, 6860, 16, 7077, 311, 387, 304, 279, 31741, 315, 1057, 27144, 1579, 2978, 11, 323, 856, 4333, 8269, 2999, 4798, 323, 358, 2751, 8041, 311, 1005, 433, 13, 1102, 574, 1093, 264, 13726, 24537, 40148, 596, 1208, 404, 1523, 1070, 11, 449, 682, 1521, 20167, 31348, 12933, 2001, 14266, 11

You can see that the result here matches the number we saw earlier.  

If you want to change the default `chunk_size` or `chunk_overlap` when you make `index`, you can use `transformations` argument in `VectorStoreIndex.from_documents()`.


### 💡 [# 14] 코드 설명 — 조각 크기를 바꿔 인덱스 다시 만들기

**한 줄 요약:** 조각을 훨씬 작게(200토큰) 자르도록 설정해서 인덱스를 새로 만듭니다.

- `SentenceSplitter(chunk_size=200, chunk_overlap=50)` : 200토큰씩, 50토큰 겹치게 자르는 도구
- `VectorStoreIndex.from_documents(documents=documents, transformations=[text_splitter])` : `transformations` 에 자르는 도구를 넣어 주면 기본 설정 대신 **내가 정한 방식**으로 자릅니다. `[ ]` 는 리스트이며 여러 처리 단계를 순서대로 넣을 수 있습니다.

> 🔰 조각이 작으면 검색이 정밀해지지만 문맥이 부족해질 수 있고, 조각이 크면 그 반대입니다. RAG 성능을 좌우하는 중요한 설정입니다.

In [16]:
# 14
### YOUR CODE HERE ###

text_splitter = SentenceSplitter(chunk_size=200, chunk_overlap=50)

index = VectorStoreIndex.from_documents(documents=documents, transformations=[text_splitter])

### 💡 [# 15] 코드 설명 — 바뀐 조각 개수 확인

**한 줄 요약:** # 9와 똑같은 코드로, 조각 크기를 줄인 뒤 조각 수가 얼마나 늘었는지 확인합니다.

- 코드는 # 9와 동일합니다. `index` 가 새로 만들어졌으므로 결과만 달라집니다.

> 🔰 조각을 작게 잘랐으니 조각 수는 훨씬 많아집니다.

In [17]:
# 15
### YOUR CODE HERE ###

node_id = index.index_struct.nodes_dict

for key, value in node_id.items():
    print(key, value)
    node_example_id = value
    break

print("The number of nodes: ", len(node_id.values()))

print(node_id.values())

b2c2b639-6dc2-4080-8dd6-931768faa7e8 b2c2b639-6dc2-4080-8dd6-931768faa7e8
The number of nodes:  144
dict_values(['b2c2b639-6dc2-4080-8dd6-931768faa7e8', '67129e71-683d-41d2-a756-03d6a0aa061d', '7f40a76c-63f1-4cbe-bef2-82c50eed0c47', '0ec4c3a6-82c0-4705-a61c-fcf383893610', '12521c6e-ccf9-43a6-afcd-249e85594ebc', 'f269919b-7e57-4c29-9aaf-22d62047172c', '824cb453-fafb-4412-bf11-9813df05ab61', '188f14a6-8e31-40f4-8afe-3cd88e7ec804', '5f55f19f-0fc7-4bd6-9014-432b7b00812d', '47c781e9-d05d-48ae-bfca-bd2c9f797157', 'a5b3573d-4b7d-407e-a9cb-6fa5c062b435', '6a38ca47-79f2-4a9a-9ab0-b6ed7252cc1f', '090917ed-dda7-4c76-90c1-7a6be162f5e9', '8ef4d9f0-71d5-4100-9225-957de1297497', 'eaaa4c33-8b2f-4f45-b58c-0bb43a3e02b0', 'eca253e8-3966-44cc-9137-aca1245d8cf9', 'a6f8fd41-c294-46da-ac6c-757e2873416c', '5695d21d-d943-4087-aba6-45f133249ce0', '684b22b8-008f-47f8-8757-7bf623906d04', 'fe02233f-1436-4cd0-b8f4-f1f5ea0deb06', 'ce7860ca-075e-43c2-aee0-814ac88a3ffe', '475b02c1-9915-43c4-b86a-d424d452b9a8', '1b2dbd

But, in this practice, we will use default `SentenceSplitter`.

### 💡 [# 16] 코드 설명 — 기본 설정으로 인덱스 되돌리기

**한 줄 요약:** 이후 실습은 기본 설정(1024토큰)으로 진행하므로 인덱스를 다시 만듭니다.

- `index = VectorStoreIndex.from_documents(documents)` : # 8과 동일. 같은 변수 이름 `index` 에 다시 담으면 **이전 값은 덮어써집니다.**

In [18]:
# 16
### YOUR CODE HERE ###

index = VectorStoreIndex.from_documents(documents)

Once the index is created, Llama Index provides a way to immediately create a query engine using the index. With the following code, we can create a query engine using the index.

### 💡 [# 17] 코드 설명 — 질문 창구(Query Engine) 만들기

**한 줄 요약:** 인덱스에서 바로 "질문하면 답해 주는" Query Engine을 만듭니다.

- `index.as_query_engine()` : 인덱스를 Query Engine으로 변환합니다. 내부적으로
  1. 질문과 비슷한 조각을 찾고(**검색**)
  2. 그 조각을 LLM에게 함께 보여 주며 답을 만들게 합니다(**생성**).

In [19]:
# 17
### YOUR CODE HERE ###

query_engine = index.as_query_engine()

### 4. Conduct Activity

Now let's actually use the query engine. We will check the results of query engine using the following questions.

#### 4-1. Answerable Questions from Data

In this activity, we will check how query engine will answer **a question related to the given data, and the information needed for the correct answer is directly provided in the data**.

#### 4-2. Unrelated Question to the Data

In this activity, we will check how query engine will answer **a question that has absolutely nothing to do with the given data**.

#### 4-3. Question Requiring Complex Reasoning

In this activity, we will check how query engine will answer **a question that is hard to answer, which means that question is related to the given data, but the information needed for the correct answer is not directly given in the data**.

### 4-1. Answerable Questions from Data
First, let's ask the query engine some questions that it can answer and some that it cannot.

For example, we could ask what program the author of this data has written about. The answer to this question is in the second sentence of the txt file.

#### Question: **What is the first programs the author tried writing?**

#### In paul_graham_essay: **The first programs I tried writing were on the IBM 1401 that our school district used for what was then called "data processing."**

Intuitively, it is expected that the query engine will generate the correct answer, which is IBM 1401. Let's verify that.

### 💡 [# 18] 코드 설명 — 정답 문장이 문서에 있는지 확인

**한 줄 요약:** 곧 물어볼 질문의 답이 되는 문장이 실제로 문서 안에 들어 있는지 검사합니다.

- `text_chunk = "..."` : 찾고 싶은 문장. 문자열 안에서 큰따옴표를 쓰려고 `\"` 로 적었습니다.
- `if A in B:` : "A가 B 안에 포함되어 있으면"이라는 뜻입니다.
- `documents[0].text` : 첫 번째 문서의 전체 글
- 포함되어 있으면 `True`, 아니면 `False` 를 출력합니다.

In [20]:
# 18
### YOUR CODE HERE ###

text_chunk = "The first programs I tried writing were on the IBM 1401 that our school district used for what was then called \"data processing.\""

if text_chunk in documents[0].text:
    print("True")
else:
    print("False")

True


### 💡 [# 19] 코드 설명 — 문서에 답이 있는 질문 해 보기

**한 줄 요약:** "저자가 처음 작성해 본 프로그램은?"이라고 Query Engine에 물어봅니다.

- `query_engine.query("질문")` : 질문을 던지고 답을 받습니다.
- `print(response)` : 답 출력

> 🔰 # 18에서 확인한 문장을 근거로 "IBM 1401 …" 같은 답이 나오면 RAG가 잘 동작한 것입니다.

In [21]:
# 19
### YOUR CODE HERE ###

response = query_engine.query("What is the first programs the author tried writing?")

print(response)

The first programs the author tried writing were on the IBM 1401 using an early version of Fortran.


<img src="https://xe.obg.co.kr/files/attach/images/4199/352/004/7e2189cf45a27f9b9cda5fef28c1dd5f.gif">

As expected, the results are accurate.  


### 4-2. Unrelated Question to the Data

This time, We will check whether the query engine performs its intended function well.  
A query engine is **a generic interface that allows users to ask questions about external data**.  
Therefore, **it should not be able to answer questions that cannot be answered through external data**.

The reason is that if the query engine can answer questions that are completely unrelated to the external data, it implies that the query engine can freely use parameterized knowledge. This, in turn, means that the answers from the query engine may contain information that is incorrect or not up-to-date.

<img src="https://media0.giphy.com/media/ANbD1CCdA3iI8/200w.gif?cid=6c09b952ooe7fzryithl047ty5npdt1xd50tlu9gcpqnzh87&ep=v1_gifs_search&rid=200w.gif&ct=g" width="200" height="200"><img src="https://st2.depositphotos.com/4421345/11492/v/950/depositphotos_114921728-stock-illustration-public-speaking-robot.jpg" width="200">

So, let’s pass on a question that has nothing to do with the original data but that the query engine’s LLM can answer.  
For example, we could use a question like this.

#### Question: **How many countries participated in the production of the space station?**
#### Answer: **15**

### 💡 [# 20] 코드 설명 — 문서와 상관없는 질문 해 보기

**한 줄 요약:** 에세이에 없는 내용(우주정거장 참여 국가 수)을 물어 Query Engine이 어떻게 반응하는지 봅니다.

- 코드는 # 19와 같고 질문만 다릅니다.

> 🔰 Query Engine은 "주어진 문서만 보고 답하라"는 지시를 받기 때문에, 문서에 없으면 "정보가 없다"는 식으로 답하는 경우가 많습니다.

In [25]:
# 20
### YOUR CODE HERE ###

response = query_engine.query("How many countries participated in the production of the space station?")

print(response)

Two countries participated in the production of the space station.


<img src = "https://img.danawa.com/images/descFiles/6/164/5163558_1_16652362092043267.gif
" width = 200>

It doesn't produce the correct answer as expected. However, we can find what kinds of LLM used in query engine through the official LlamaIndex documentation.

Actually, Llamaindex uses `gpt-3.5-turbo` as LLM of response synthesizer. So, we can ask `gpt-3.5-turbo` to answer the question.

Let's check this out with a simple code. We can use the following code to separately utilize the OpenAI LLM.  



### 💡 [# 21] 코드 설명 — LLM에게 직접 묻는 함수 만들기

**한 줄 요약:** 문서 검색 없이 ChatGPT에게 바로 질문하고 답을 돌려받는 함수 `generate_answer` 를 정의합니다.

- `def generate_answer(question):` : **함수 정의**입니다. `question` 은 함수가 받는 입력(매개변수)입니다. 이 셀을 실행해도 아무 결과가 안 보이는 것이 정상입니다(정의만 함).
- `messages = [ {"role": "user", "content": question} ]` : 대화 내용을 담은 리스트. `role` 은 누가 말했는지(`user`=사용자), `content` 는 말한 내용입니다.
- `openai.chat.completions.create(...)` : OpenAI 서버에 요청을 보냅니다.
  - `model="gpt-3.5-turbo"` : 사용할 모델
  - `temperature=0` : 답변의 무작위성. 0이면 매번 거의 같은 답, 높을수록 다양하고 창의적인 답
- `return response.choices[0].message.content` : 응답에서 **답변 글만** 꺼내 함수 밖으로 돌려줍니다.

In [26]:
# 21
### YOUR CODE HERE ###

def generate_answer(question):
    messages = [
        {
            "role": "user",
            "content": question,
        },
    ]
    response = openai.chat.completions.create(
        model="gpt-3.5-turbo",
        temperature=0,
        messages=messages,
    )

    return response.choices[0].message.content

### 💡 [# 22] 코드 설명 — 같은 질문을 LLM에 직접 하기

**한 줄 요약:** # 20의 질문을 방금 만든 함수로 LLM에게 직접 물어봅니다.

- `generate_answer(question)` : 함수 호출. 결과(답변 글)를 `print` 로 바로 출력합니다.

> 🔰 LLM은 원래 알고 있는 지식으로 답할 수 있습니다. **Query Engine(문서 기반)** 과 **LLM 단독** 의 차이를 비교해 보세요.

In [27]:
# 22
### YOUR CODE HERE ###

question = "How many countries participated in the production of the space station?"

print(generate_answer(question))

The International Space Station (ISS) is a collaborative project involving five space agencies from 15 countries. These countries are the United States, Russia, Japan, Canada, and 11 European countries represented by the European Space Agency.


In fact, as in the example above, OpenAI LLM already knows the answer to this question.  
However, since the relevant information cannot be found in the txt file that query engine can refer to here, it does not generate a correct answer for a question that is already known.  

<img src="https://i.imgur.com/sxSRQks.png">

From this, we can understand that LLM in the response synthesizer of the query engine uses the information from the retrieved passages produced by the retriever, excluding its own parametrized knowledge.  

### 4-3. Question Requiring Complex Reasoning

Now, let’s explore more deeply. There is a question related to the data that is hard to answer. We will see how the query engine responds to such questions.

The question we will ask in this section is who is the author of the `paul_graham_essay.txt`. Becuase the original text file does not mention the author's name directly, this is a complex question.

Question: **Who is the author?**

Let's see whether the writer's name is actually not in the original text file or not.

### 💡 [# 23] 코드 설명 — 파일에서 'Paul Graham'이 나오는 줄 찾기

**한 줄 요약:** 에세이 파일에서 저자 이름이 언급된 줄을 찾아 그 줄의 마지막 문장을 출력합니다.

- `textwrap.TextWrapper(width=80)` : 한 줄을 80자 폭으로 줄바꿈해 주는 도구
- `for line in file:` : 파일을 **한 줄씩** 끝까지 읽습니다.
- `line.lower()` : 모두 소문자로 바꿉니다. 대소문자 구분 없이 찾기 위해서입니다.
- `if 'paul graham' in line.lower():` : 그 줄에 이름이 있으면
- `wrapper.fill(line.strip())` : 80자 폭으로 정리
- `.split('.')[-1]` : 마침표로 잘라 나눈 것 중 **마지막 조각**(`[-1]` 은 "뒤에서 첫 번째")만 출력

In [28]:
# 23
### YOUR CODE HERE ###

wrapper = textwrap.TextWrapper(width=80)

with open(path_to_txt, 'r', encoding='utf-8') as file:
    for line in file:
        if 'paul graham' in line.lower():
            formatted_text = wrapper.fill(line.strip())
            print(formatted_text.split('.')[-1])


Occasionally after wrestling for hours with some gruesome bug I'd check Twitter
or HN and see someone asking "Does Paul Graham still code?"


This is the only sentence in the original txt file that mentions 'Paul Graham'.  
Now, let's read it ourselves and consider whether it actually specifies the author's name.

<br/>

Someone might infer that the author's name is Paul Graham from this sentence, but I think it's not certain

In my opinion, this sentence does not explicitly state the author's name, so it does not provide enough useful information

<br/>

However, opinions on this may vary, so let's verify it ourselves.


### 💡 [# 24] 코드 설명 — 추론이 필요한 질문 해 보기

**한 줄 요약:** "저자가 누구인가?"처럼 문서에 직접 써 있지 않은 질문을 던져 봅니다.

- `response_complex = query_engine.query("Who is the author?")` : 질문 후 답 저장

> 🔰 문서에 "저자는 Paul Graham"이라고 적혀 있지 않으므로 여러 단서를 조합해야 하는 어려운 질문입니다.

In [29]:
# 24
### YOUR CODE HERE ###

response_complex = query_engine.query("Who is the author?")

print(response_complex)

Paul Graham


<img src="https://i3.ruliweb.net/ori/21/07/30/17af799f631524cc2.gif">

Hmm... The query engine answers better than we think.  
Then, let's see what data were referred to by the query engine.
<br/>

<img src="https://i.imgur.com/sxSRQks.png">

The query engine made using `as_query_engine` uses a retriever generated directly from the index by using `as_retriever` method.   
Therefore, if we look at the results of this retriever, we can see which retrieved passages the query engine has seen.


### 💡 [# 25] 코드 설명 — 검색 단계만 따로 실행해 보기

**한 줄 요약:** 답을 만들지 않고, 질문에 대해 **어떤 조각이 검색되는지만** 확인합니다.

- `index.as_retriever()` : 인덱스에서 "검색 담당(Retriever)"만 꺼냅니다.
- `retriever.retrieve("질문")` : 질문과 가장 비슷한 조각들을 돌려줍니다(기본 2개).
- 반복문으로 각 조각의 `.text` 를 출력합니다.

> 🔰 RAG의 답이 이상할 때는 이렇게 **"무엇이 검색되었나"부터 확인**하는 것이 디버깅의 기본입니다.

In [30]:
# 25
### YOUR CODE HERE ###

retriever = index.as_retriever()

ret_passages = retriever.retrieve("Who is the author?")

for i in range(len(ret_passages)):
  print("###Retrieved Passage\n", ret_passages[i].text)
  print("\n\n\n")

###Retrieved Passage
 After I moved to New York I became her de facto studio assistant.

She liked to paint on big, square canvases, 4 to 5 feet on a side. One day in late 1994 as I was stretching one of these monsters there was something on the radio about a famous fund manager. He wasn't that much older than me, and was super rich. The thought suddenly occurred to me: why don't I become rich? Then I'll be able to work on whatever I want.

Meanwhile I'd been hearing more and more about this new thing called the World Wide Web. Robert Morris showed it to me when I visited him in Cambridge, where he was now in grad school at Harvard. It seemed to me that the web would be a big deal. I'd seen what graphical user interfaces had done for the popularity of microcomputers. It seemed like the web would do the same for the internet.

If I wanted to get rich, here was the next train leaving the station. I was right about that part. What I got wrong was the idea. I decided we should start a comp

This is quiet lone passages. So, it is hard to check whether there is any evidence of query engine's response or not.  
Can you find any supporting evidence of the question?  

Let's see if 'Paul Graham' is in there.



### 💡 [# 26] 코드 설명 — 검색된 조각에 이름이 들어 있는지 검사

**한 줄 요약:** 검색된 조각들을 하나로 이어 붙인 뒤 'Paul Graham'이라는 글자가 있는지 확인합니다.

- `passages = ''` : 빈 문자열로 시작
- `passages += ...` : 기존 글 뒤에 **이어 붙이기**. (`a += b` 는 `a = a + b` 와 같습니다.)
- `if 'paul graham' in passages.lower():` : 소문자로 바꿔 포함 여부 검사 후 결과 문장 출력

In [31]:
# 26
### YOUR CODE HERE ###

passages = ''

for i in range(len(ret_passages)):
  passages += ret_passages[i].text

if 'paul graham' in passages.lower():
    print("The word 'Paul Graham' is found in the passages.")
else:
    print("The word 'Paul Graham' is not found in the passages.")

The word 'Paul Graham' is not found in the passages.


<img src="https://i2.ruliweb.net/ori/21/07/24/17ad8e756ba54811a.gif" width="400">

???

Retriever couldn't find the only sentence with the word Paul Graham on it. Then how did query engine know that the answer was Paul Graham?

To verify this, let’s using the query engine’s LLM as we did in the previous section and ask the same question. We will use a prompt as similar as possible to the one used in LlamaIndex response synthesizer, but with an additional instruction to provide the reason for the LLM’s answer.  

The prompt used by the original Response Synthesizer is as follows.


<img src = "https://i.imgur.com/ZXUDAJi.jpeg" width="450">  


<br/>

Therefore, we will write the code as shown below to check the results of the LLM.



### 💡 [# 27] 코드 설명 — 검색 결과로 직접 프롬프트 만들어 LLM에 묻기

**한 줄 요약:** Query Engine이 속에서 하는 일을 손으로 재현합니다. 검색된 글을 프롬프트에 넣고 "답과 그 이유"를 물어봅니다.

- `ret_context` : 검색된 조각을 모두 이어 붙인 글
- `question = f"""..."""` : 따옴표 세 개(`"""`)는 **여러 줄 문자열**입니다. `f` 가 붙어 있어 `{ret_context}` 자리에 실제 글이 들어갑니다.
- 프롬프트 내용: "아래 문맥만 사용하고 사전 지식은 쓰지 말고, 답과 이유를 말하라"
- `generate_answer(question)` : # 21에서 만든 함수로 LLM 호출

> 🔰 이것이 RAG의 핵심 원리입니다: **검색한 글 + 질문 → 하나의 프롬프트 → LLM**

In [32]:
# 27
### YOUR CODE HERE ###

ret_context = ""
for ret_result in ret_passages:
  ret_context += ret_result.text

question = f"""Context information is below.
---------------------
{ret_context}
---------------------
Given the context information and not prior knowledge,
answer the query and the reasons of answer.
Query: Who is the author?
Answer:
"""

print(generate_answer(question))

The author of the text is Paul Graham. This can be inferred from the personal experiences and insights shared throughout the text, such as his involvement in starting Viaweb, his interest in AI, and his experiences at Cornell and Harvard.


Well... it seems like query engine can **capture useful information not only in direct evidence but also in indirect content by using its prior knowledge, i.e., parametrized knowledge**.  

Therefore, we can consider that the query engine has the ability to provide correct answers even when the given query is not explicitly answered in the data.  

However, the problem is that **without such an explanation, the reasoning process cannot be trusted** because it is impossible to know whether the LLM used reliable information in its reasoning.

<br/>

Here’s what we can infer from this:

the query engine has the capability to answer not only questions directly addressed in the data **but also questions that can be indirectly inferred from the data.**  
However, in such cases, **the reliability of the answers is compromised because we cannot verify the information used and the reasoning steps.**   

To address this issue, we can use one of the following two methods.  


#### **1.** Force the response synthesizer to utilize only the information in a given retrieved passages.
#### **2.** When the response synthesizer generates a summary, make sure that the reason is also generated.

## II. Use Database Management Method

Now, let’s consider a scenario. If we have new data and want to create a query engine using it, we could recreate the `document`, `index`, and `query engine` as we did above.

However, this is quite cumbersome and time-consuming. In addition, if you want to insert new data so that query engine uses that data, the above method is very expansive because a new index must be generated for every inserts operation.

So, Llama Index is providing the following functions to manage data smoothly.

### 1. Insert

We can insert new data to the existing query engine.

To check this, we will ask a question which can't get useful information from the data before inserting new data, and see what happens after inserting data.

In this section, the following data and question will be used.

Inserted Data: **"Natural diamonds were (and are) formed (thousands of million years ago) in the upper mantle of Earth in metallic melts at temperatures of 900–1,400 °C and at pressures of 5–6 GPa."**  
Question: **What are the temperature and air pressure conditions under which natural diamonds are produced?**


### 💡 [# 28] 코드 설명 — 새 정보 준비 + 넣기 전 질문해 보기

**한 줄 요약:** 다이아몬드 생성 조건에 대한 새 글과 질문을 준비하고, 아직 그 글을 넣지 않은 상태에서 먼저 물어봅니다.

- `data_text` : 나중에 인덱스에 추가할 새 정보(900–1,400 °C, 5–6 GPa)
- `question` : 그 정보에 대한 질문
- `res = query_engine.query(question)` : 현재 인덱스에는 에세이만 있으므로 제대로 답하지 못할 것입니다.

In [33]:
# 28
### YOUR CODE HERE ###

data_text = "Natural diamonds were (and are) formed (thousands of million years ago) in the upper mantle of Earth in metallic melts at temperatures of 900–1,400 °C and at pressures of 5–6 GPa."
question = "What are the temperature and air pressure conditions under which natural diamonds are produced?"

res = query_engine.query(question)

print(res)

Natural diamonds are formed under high temperature and pressure conditions deep within the Earth's mantle.


<img src = "https://i.pinimg.com/originals/15/8b/ed/158bed9819e4fccf7e18a5eeeaf79c6b.png" width = 200>

Hmm… the query engine answers quite well. It seems the response synthesizer is using its parameterized knowledge. However, there are no specific details like 900–1,400°C and pressures of 5–6 GPa.

Anyway, it's time to insert the data.



### 💡 [# 29] 코드 설명 — 인덱스에 새 문서 추가(Insert)

**한 줄 요약:** 새 글을 `Document` 로 감싸 인덱스에 넣고, Query Engine을 새로 만듭니다.

- `Document(text=data_text, id_="new_doc_id")` : 글을 문서 객체로 만듭니다. `id_` 는 이 문서의 **이름표**로, 나중에 수정·삭제할 때 씁니다.
- `index.insert(docu)` : 인덱스에 추가(조각내기 + 임베딩이 자동 수행됨)
- `new_query_engine = index.as_query_engine()` : 갱신된 인덱스로 질문 창구 다시 생성

In [34]:
# 29
### YOUR CODE HERE ###

docu = Document(text=data_text, id_="new_doc_id")

index.insert(docu)

new_query_engine = index.as_query_engine()

### 💡 [# 30] 코드 설명 — 추가 후 다시 질문

**한 줄 요약:** 같은 질문을 다시 해서 이제는 답이 나오는지 확인합니다.

- `new_query_engine.query(question)` : 이번에는 "900–1,400 °C, 5–6 GPa"가 들어간 답이 나와야 합니다.

In [35]:
# 30
### YOUR CODE HERE ###

new_res = new_query_engine.query(question)

print(new_res)

Natural diamonds are formed in metallic melts at temperatures ranging from 900 to 1,400 degrees Celsius and at pressures of 5 to 6 gigapascals.


It's as expected. Query engine is generating an appropriate answer with detail using information in new `Document` object.

Then, what happens if we change the information in that data?

### 2. Update

If a Document is already present within an index, you can "update" a `Document` with the same doc `id_`. The `update_ref_doc` method receives a single `Document` object and updates the value of data that has the same `id`.

Or you can "refresh" all document at once. The `refresh_ref_doc` method finds the `id` of the `Document` object that came into the input, updates the data with the same `id`, or inserts the data if there is no other data with the same `id`

**`update_ref_doc`**  
**Input**: single `Document` object  
**Output**: None

**`refresh_ref_doc`**  
**Inpu**t: list of `Document` object  
**Output**: a boolean list, indicating which documents in the input have been refreshed in the index.

You will see `False` in that boolean list if text of document doesn't change.  

In both methods, we can set `delete_from_docstore` to `True` or `False`. A detailed description of this will be given in Delete.


<br/>

Let's check by changing the base question a little bit.  

I changed the information about temperature and air pressure in the question, so check it out for yourself.  



### 💡 [# 31] 코드 설명 — 문서 내용 수정(Update)

**한 줄 요약:** 넣어 둔 문서의 숫자를 바꾼 뒤, 인덱스에 그 변경을 반영합니다.

- `docu.set_content(value="...")` : 문서 내용을 새 글(2,000–6,000 °C, 8–9 GPa)로 교체
- `index.update_ref_doc(docu, update_kwargs={...})` : 같은 ID의 옛 문서를 **지우고 새 내용으로 다시 넣습니다.** `delete_from_docstore: True` 는 문서 저장소에서도 옛 내용을 완전히 지우라는 옵션입니다.
- `print(output)` : 이 함수는 돌려주는 값이 없어 `None` 이 출력됩니다(정상).
- 이후 Query Engine을 새로 만들어 같은 질문 → 바뀐 숫자로 답하는지 확인

In [36]:
# 31
### YOUR CODE HERE ###

docu.set_content(value="Natural diamonds were (and are) formed (thousands of million years ago) in the upper mantle of Earth in metallic melts at temperatures of 2,000–6,000 °C and at pressures of 8–9 GPa.")

output = index.update_ref_doc(
    docu,
    update_kwargs={"delete_kwargs": {"delete_from_docstore": True}},
)

print(output)

query_engine_update = index.as_query_engine()

res_update = query_engine_update.query(question)

print(res_update)

None
Natural diamonds are formed in metallic melts at temperatures ranging from 2,000 to 6,000 degrees Celsius and at pressures between 8 to 9 gigapascals.


### 💡 [# 32] 코드 설명 — 여러 문서 한꺼번에 새로고침(Refresh)

**한 줄 요약:** 내용을 또 바꾼 뒤 `refresh_ref_docs` 로 "바뀐 문서만" 골라 갱신합니다.

- `docu.set_content(...)` : 숫자를 6,000–8,000 °C, 12–15 GPa로 다시 변경
- `index.refresh_ref_docs([docu])` : 문서 **리스트**를 받아, 내용이 바뀐 문서만 갱신하고 새 문서는 추가합니다.
- `print(output)` : `[True]` 처럼 문서별로 "갱신되었는지" 여부가 리스트로 나옵니다.

> 🔰 `update_ref_doc` 는 문서 1개를 무조건 교체, `refresh_ref_docs` 는 여러 개 중 **바뀐 것만** 교체한다는 차이가 있습니다.

In [37]:
# 32
### YOUR CODE HERE ###

docu.set_content(value="Natural diamonds were (and are) formed (thousands of million years ago) in the upper mantle of Earth in metallic melts at temperatures of 6,000–8,000 °C and at pressures of 12–15 GPa.")

output = index.refresh_ref_docs(
    [docu]
)

print(output)

query_engine_refresh = index.as_query_engine()

res_refresh = query_engine_refresh.query(question)

print(res_refresh)

[True]
Natural diamonds are formed in metallic melts at temperatures ranging from 6,000 to 8,000 degrees Celsius and at pressures between 12 to 15 gigapascals.


You can check list of doc_id in the index. So, if you want more experience with other data, you can check whether your data is inserted properly by looking at that list

### 💡 [# 33] 코드 설명 — 인덱스에 들어 있는 문서 ID 목록 보기

**한 줄 요약:** 현재 인덱스에 어떤 문서들이 들어 있는지 ID로 확인합니다.

- `index.ref_doc_info` : "문서 ID → 문서 정보" 사전
- `.keys()` : 사전의 키(여기서는 문서 ID)만 모아 보여 줍니다.

> 🔰 에세이 문서의 ID와 우리가 넣은 `new_doc_id` 두 개가 보여야 합니다.

In [38]:
# 33
### YOUR CODE HERE ###

print(index.ref_doc_info.keys())

dict_keys(['3612a786-4b39-4c8f-ad7c-5f56cfa9b4ec', 'new_doc_id'])


### 3. Delete

We wrapped the new information above with a `Document` object and inserted it into the index.

At this time, we were able to specify the `Document` object using `doc_id`. What this doc_id is can be confirmed as follows.

### 💡 [# 34] 코드 설명 — 지울 문서의 ID 꺼내기

**한 줄 요약:** 우리가 추가한 문서의 ID를 변수에 담아 출력합니다.

- `id = docu.doc_id` : 문서의 ID(`new_doc_id`)를 `id` 변수에 저장

> 🔰 `id` 는 사실 파이썬에 원래 있는 함수 이름입니다. 변수 이름으로 써도 동작은 하지만 원래 기능이 가려지므로, 실무에서는 `doc_id` 같은 이름을 쓰는 것이 좋습니다.

In [39]:
# 34
### YOUR CODE HERE ###

id = docu.doc_id

print(id)

new_doc_id


Using this, we can change the contents of data with the `doc_id` using the `delete` method of the index.

Let's delete the data related to a diamond, and see how query engine answers for the same question.

Through the value of `delete_from_docstroe`, it is possible to determine whether the data with that id actually disappears on the database, or disappears only on the index and remains in the database. Even if it disappears only on the index, the information of the data cannot be used by the query engine.



### 💡 [# 35] 코드 설명 — 문서 삭제(Delete)

**한 줄 요약:** 해당 ID의 문서를 인덱스에서 지우고, 같은 질문을 다시 해 봅니다.

- `index.delete_ref_doc(id, delete_from_docstore=True)` : 그 문서와 거기서 나온 조각들을 모두 삭제
- Query Engine을 다시 만들고 질문 → 이제는 다이아몬드 정보가 없으므로 다시 제대로 답하지 못해야 합니다.

In [40]:
# 35
### YOUR CODE HERE ###

index.delete_ref_doc(id, delete_from_docstore=True)

query_engine_delete = index.as_query_engine()

res_delete = query_engine_delete.query(question)

print(res_delete)

Natural diamonds are produced under high temperature and pressure conditions deep within the Earth's mantle.


If the answer still contains very details of condition of natural diamonds, check the index whether that data was deleted properly.

The result of `index.ref_doc_info.keys()` has to contain only one doc_id.

### 💡 [# 36] 코드 설명 — 삭제 확인

**한 줄 요약:** 문서 ID 목록을 다시 출력해 `new_doc_id` 가 사라졌는지 확인합니다.

- # 33과 같은 코드입니다. 이제 에세이 문서 ID 하나만 남아 있어야 합니다.

In [41]:
# 36
### YOUR CODE HERE ###

print(index.ref_doc_info.keys())

dict_keys(['3612a786-4b39-4c8f-ad7c-5f56cfa9b4ec'])


## III. Make a Custom Query Engine

In this section, we will define the existing query engine in more detail and modify its internal components to create the query engine we desire.
<br/>

The query engine can vary in its usage depending on the task requested after passing the retrieved documents to the LLM in response synthesizer. For instance, it can be tasked with summarizing the documents or directly answering the query.

Since the query engine can read through extremely long documents that exceed the LLM’s context limit, it can have a significant impact on RAG, depending on how it is used. Therefore, choosing which query engine to use is an important design choice when designing RAG.

This section is composed of the following four stages:
 <br/>
#### 1. Defining the query engine in detail
#### 2. Creating a custom query engine


### 1. Defining the query engine in detail

In this section, we will define the retriever and response synthesizer, which are internal components of the query engine, and use them to define a class that performs the role of the query engine.

### 💡 [# 37] 코드 설명 — 기본형 Query Engine을 직접 정의하기

**한 줄 요약:** "검색 → 답변 합성"이라는 Query Engine의 동작을 **클래스**로 직접 작성합니다.

- `class StandardQueryEngine(CustomQueryEngine):` : **클래스**는 "설계도"입니다. 괄호 안은 부모 클래스로, LlamaIndex가 제공하는 `CustomQueryEngine` 의 기능을 **물려받아(상속)** 필요한 부분만 채웁니다.
- `retriever: BaseRetriever` / `response_synthesizer: BaseSynthesizer` : 이 엔진이 가질 부품 두 개와 그 **종류(타입)** 를 선언합니다.
- `def custom_query(self, query_str: str):` : 질문이 들어왔을 때 실행될 함수. `self` 는 "나 자신(이 엔진)"을 가리키며, `self.retriever` 처럼 자기 부품을 쓸 때 사용합니다.
  - `self.retriever.retrieve(query_str)` : ① 관련 조각 검색
  - `self.response_synthesizer.synthesize(query_str, nodes)` : ② 질문 + 조각으로 답변 생성
  - `return response_obj` : 답 돌려주기

In [42]:
# 37
### YOUR CODE HERE ###

class StandardQueryEngine(CustomQueryEngine):
    retriever: BaseRetriever
    response_synthesizer: BaseSynthesizer

    def custom_query(self, query_str: str):
        nodes = self.retriever.retrieve(query_str)
        response_obj = self.response_synthesizer.synthesize(query_str, nodes)
        return response_obj

Initialize the retriever and response synthesizer, and then verify that the class functions correctly.


### 💡 [# 38] 코드 설명 — 직접 만든 엔진 조립 및 테스트

**한 줄 요약:** 부품(검색기, 합성기)을 준비해 # 37의 설계도로 엔진을 만들고 질문합니다.

- `retriever = index.as_retriever()` : 검색 담당
- `get_response_synthesizer(response_mode="compact")` : 답변 합성 담당. `compact` 는 검색된 조각을 가능한 한 **한 프롬프트에 모아** LLM 호출 횟수를 줄이는 방식입니다.
- `StandardQueryEngine(retriever=..., response_synthesizer=...)` : 설계도로 실제 물건(**객체**)을 만듭니다.
- `query_engine.query("...")` : `query` 를 부르면 내부에서 우리가 쓴 `custom_query` 가 실행됩니다.
- `str(response)` : 응답 객체를 글자로 변환

In [43]:
# 38
### YOUR CODE HERE ###

retriever = index.as_retriever()
synthesizer = get_response_synthesizer(response_mode="compact")
query_engine = StandardQueryEngine(
    retriever=retriever, response_synthesizer=synthesizer
)

response = query_engine.query("What did the author do growing up?")
print(str(response))

The author focused on writing and programming before college.


### 2. Creating a custom query engine

Now, let’s modify the query engine so that we can pass the desired task to the response synthesizer through a prompt.  

While there are many aspects of the retriever that can also be adjusted, in this exercise, we will focus on changing the content passed to the response synthesizer to experience more noticeable changes

from llama_index.core import PromptTemplate


### 💡 [# 39] 코드 설명 — 프롬프트 틀 두 가지 만들기

**한 줄 요약:** LLM에게 보낼 지시문의 틀을 "질문에 답하기"용과 "20단어 이내 요약"용 두 가지로 만듭니다.

- `PromptTemplate("...")` : 빈칸이 있는 문장 틀. `{context_str}` 에는 검색된 글이, `{query_str}` 에는 질문이 나중에 채워집니다.
- 괄호 안에 문자열 여러 개를 줄줄이 적으면 파이썬이 **하나로 이어 붙입니다.**
- `simple_qa_prompt` : 문맥만 보고 질문에 답하라는 틀
- `short_sum_prompt` : 문맥을 20단어 이내로 요약하라는 틀

> 🔰 같은 검색 결과라도 **프롬프트를 바꾸면 결과물의 성격이 달라집니다.**

In [44]:
# 39
### YOUR CODE HERE ###

simple_qa_prompt = PromptTemplate(
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and not prior knowledge, "
    "answer the query.\n"
    "Query: {query_str}\n"
    "Answer: "
)

short_sum_prompt = PromptTemplate(
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and not prior knowledge, "
    "give the summary of context, less than 20 words.\n"
    "Query: {query_str}\n"
    "Summary: "
)

### 💡 [# 40] 코드 설명 — 프롬프트를 바꿀 수 있는 맞춤 엔진 정의

**한 줄 요약:** 합성기를 쓰지 않고, 내가 정한 프롬프트로 LLM을 직접 호출하는 Query Engine을 만듭니다.

- `from llama_index.llms.openai import OpenAI` : LlamaIndex용 OpenAI LLM 도구. (# 2에서 불러온 `openai` 의 `OpenAI` 와 이름이 같아 **이 줄 이후로는 이쪽이 사용됩니다.**)
- 부품 선언: `retriever`, `response_synthesizer`, `llm`, `qa_prompt`. `qa_prompt: PromptTemplate = simple_qa_prompt` 는 **기본값**을 정해 둔 것입니다.
- `custom_query` 안에서:
  - `nodes = self.retriever.retrieve(query_str)` : 조각 검색
  - `[n.node.get_content() for n in nodes]` : **리스트 컴프리헨션**. "nodes의 각 n에서 글을 꺼내 새 리스트를 만들라"는 한 줄 반복문입니다.
  - `"\n\n".join(리스트)` : 리스트의 글들을 빈 줄로 구분해 하나로 합칩니다.
  - `self.qa_prompt.format(context_str=..., query_str=...)` : 틀의 빈칸 채우기
  - `self.llm.complete(...)` : 완성된 프롬프트를 LLM에 보내 답 받기

> 🔰 `response_synthesizer` 는 선언만 하고 실제로는 쓰지 않습니다. 대신 `llm.complete` 로 직접 답을 만듭니다.

In [45]:
# 40
### YOUR CODE HERE ###

from llama_index.llms.openai import OpenAI

class OurCustomQueryEngine(CustomQueryEngine):

    retriever: BaseRetriever
    response_synthesizer: BaseSynthesizer
    llm: OpenAI
    qa_prompt: PromptTemplate = simple_qa_prompt

    def custom_query(self, query_str: str):
        nodes = self.retriever.retrieve(query_str)

        context_str = "\n\n".join([n.node.get_content() for n in nodes])
        response = self.llm.complete(
            self.qa_prompt.format(context_str=context_str, query_str=query_str)
        )

        return str(response)

Now, let’s check if it truly works as we intended.


### 💡 [# 41] 코드 설명 — 맞춤 엔진 조립 및 테스트

**한 줄 요약:** LLM을 준비해 맞춤 엔진을 만들고 질문합니다.

- `llm = OpenAI(model="gpt-3.5-turbo")` : 사용할 LLM 지정
- `OurCustomQueryEngine(retriever=..., response_synthesizer=..., llm=llm, qa_prompt=simple_qa_prompt)` : 부품을 넣어 엔진 생성
- `query_engine.query("What did the author do growing up?")` : 질문 후 출력

> 🔰 **직접 해 보기:** `qa_prompt=short_sum_prompt` 로 바꿔 실행하면 답변 대신 20단어 이내 요약이 나옵니다.

In [46]:
# 41
### YOUR CODE HERE ###

llm = OpenAI(model="gpt-3.5-turbo")

query_engine = OurCustomQueryEngine(
    retriever=retriever,
    response_synthesizer=synthesizer,
    llm=llm,
    qa_prompt=simple_qa_prompt,
)

response = query_engine.query("What did the author do growing up?")
print(str(response))

Growing up, the author worked on writing short stories and programming. They started programming on an IBM 1401 in 9th grade, using an early version of Fortran. Later on, they got a microcomputer and started programming on a TRS-80, writing simple games and a word processor. They also applied to art schools and eventually attended RISD for a BFA program.


### 💡 [# 42] 코드 설명 — 빈 셀

번호만 있고 코드가 없는 **연습용 빈 셀**입니다. 위에서 배운 내용을 자유롭게 실험해 보세요. (예: 질문 바꿔 보기, `chunk_size` 바꿔 보기)

In [ ]:
# 42
